# Conexión A2A entre agentes

Notebook para conectar dos agentes que ya están levantados y expuestos con Cloudflare, descubrir sus Agent Cards y hacerlos conversar (A propone, B mejora, A integra).

**Requisitos previos:**
- Tu agente corriendo (`langgraph dev --port 2025`) y expuesto con `cloudflared tunnel --url http://localhost:2025`.
- Tu compañero igual, con su propia URL de Cloudflare.
- Cada agente con un nombre de grafo distinto.

La comunicación A2A se hace por **JSON-RPC directo** (`message/send`) sobre el endpoint `/a2a/<assistant_id>`, para ser compatible con cualquier versión del servidor de LangGraph.

Ejecuta las celdas en orden.

## 1. Pega aquí las URLs de Cloudflare

Una línea por agente. La primera suele ser la tuya; agrega la de cada compañero.

In [1]:
SERVIDORES = [
    "https://nursing-speed-fix-validity.trycloudflare.com",        # tu agente (Bryhan)
    "https://topics-seeing-echo-bench.trycloudflare.com",  # agente del compañero (Carlos)
]

# Reto en común que resolverán los dos agentes.
RETO = "Organicemos una actividad de bienvenida para estudiantes nuevos."

print("Servidores a conectar:")
for s in SERVIDORES:
    print(" -", s)

Servidores a conectar:
 - https://nursing-speed-fix-validity.trycloudflare.com
 - https://topics-seeing-echo-bench.trycloudflare.com


## 2. Dependencias

Usa el cliente de LangGraph (para listar asistentes) y `httpx` (para enviar mensajes por JSON-RPC).

In [2]:
import json
import uuid

import httpx
from langgraph_sdk import get_client

def mostrar(valor):
    print(json.dumps(valor, indent=2, ensure_ascii=False, default=str))

def _base(url):
    url = str(url).strip().rstrip("/")
    if not url.startswith(("http://", "https://")):
        raise ValueError(f"URL inválida: {url!r}")
    return url

print("Dependencias cargadas.")

Dependencias cargadas.


## 3. Descubrir los agentes

Consulta cada servidor, obtiene su `assistant_id` y descarga su Agent Card (`/.well-known/agent-card.json`). Guarda para cada agente su URL base y su `assistant_id`. Debe imprimir el nombre de cada agente disponible.

In [3]:
# Cliente httpx compartido para todas las llamadas A2A.
if "http_a2a" in globals():
    await http_a2a.aclose()
http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)

catalogo = []
agentes_por_nombre = {}   # nombre -> {"base": url, "aid": assistant_id}
tarjetas = {}             # nombre -> Agent Card (JSON) descargada por A2A
ids_vistos = set()

for servidor in SERVIDORES:
    base = _base(servidor)
    if base.startswith("https://TU-URL") or "URL-DEL-COMPANERO" in base:
        print(f"Omitido (placeholder sin reemplazar): {base}")
        continue
    try:
        async with get_client(url=base, api_key=None, timeout=30) as remoto:
            asistentes = await remoto.assistants.search(limit=100, response_format="array")
    except Exception as error:
        print(f"No pude consultar {base}: {type(error).__name__}: {error}")
        continue

    for asistente in asistentes:
        aid = asistente["assistant_id"]
        if aid in ids_vistos:
            raise ValueError(f"Assistant ID repetido: {aid}. Usa nombres de grafo distintos.")
        # Descarga la Agent Card por A2A (evidencia de la tarjeta).
        rc = await http_a2a.get(f"{base}/.well-known/agent-card.json", params={"assistant_id": aid})
        card = rc.json()
        nombre = card.get("name", aid)
        if nombre in agentes_por_nombre:
            raise ValueError(f"Nombre de Agent Card repetido: {nombre}. Cambia el nombre de la tarjeta.")
        agentes_por_nombre[nombre] = {"base": base, "aid": aid}
        tarjetas[nombre] = card
        ids_vistos.add(aid)
        habilidades = ", ".join(s.get("name", "") for s in card.get("skills", []))
        catalogo.append(f"- {nombre}: {card.get('description', '')}  (habilidades: {habilidades})")

print("Agentes disponibles:", list(agentes_por_nombre))
print()
print("\n".join(catalogo))

Agentes disponibles: ['agente_bryhan', 'agente_czz']

- agente_bryhan: agente_bryhan assistant  (habilidades: agente_bryhan Capabilities)
- agente_czz: agente_czz assistant  (habilidades: agente_czz Capabilities)


### 3.0 Tarjeta descargada por A2A

Muestra la Agent Card completa de cada agente, tal como se descargó del endpoint `/.well-known/agent-card.json`.

In [4]:
for nombre, card in tarjetas.items():
    print(f"===== Agent Card: {nombre} =====")
    mostrar(card)
    print()

===== Agent Card: agente_bryhan =====
{
  "name": "agente_bryhan",
  "description": "agente_bryhan assistant",
  "url": "http://127.0.0.1:2025/a2a/a4795f3b-403a-5782-ae56-443223f44e86",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2025/a2a/a4795f3b-403a-5782-ae56-443223f44e86",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "required": false,
        "params": {
          "header": "LangGraph-A2A-History-Scope",
          "values": [
            "context",
            "task"
          ],
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ]
        }
      }
  

## 3.1 Encontrar el Id

El servidor de LangGraph asigna a cada grafo un `assistant_id` (UUID). Es el identificador con el que se arma el endpoint A2A (`/a2a/<assistant_id>`) y la URL de la Agent Card. Aquí se muestra el de cada agente descubierto; el primero es el de tu agente.

In [5]:
# Encontrar el Id: muestra el assistant_id (UUID) de cada agente del catálogo.
if not agentes_por_nombre:
    print("No hay agentes descubiertos. Ejecuta antes la celda 'Descubrir los agentes' con URLs reales.")
else:
    for nombre, info in agentes_por_nombre.items():
        print(f"{nombre}: assistant_id = {info['aid']}")

    # El primero del catálogo corresponde a tu agente (primera URL de SERVIDORES).
    MI_NOMBRE = next(iter(agentes_por_nombre))
    MI_ASSISTANT_ID = agentes_por_nombre[MI_NOMBRE]["aid"]
    print()
    print("MI agente:", MI_NOMBRE)
    print("MI assistant_id:", MI_ASSISTANT_ID)

agente_bryhan: assistant_id = a4795f3b-403a-5782-ae56-443223f44e86
agente_czz: assistant_id = 3d58b072-566f-59e7-90f2-8fb84b2a11fe

MI agente: agente_bryhan
MI assistant_id: a4795f3b-403a-5782-ae56-443223f44e86


## 4. Función para enviar un mensaje a un agente

Envía un mensaje por A2A (`message/send` vía JSON-RPC) y conserva el `contextId` (hilo de conversación del agente remoto). Lee la respuesta como JSON, sin depender del esquema protobuf del SDK, de modo que funciona con cualquier versión del servidor.

In [6]:
def _extraer_texto(result: dict) -> str:
    '''Extrae el texto de la respuesta del agente desde el task A2A (status, artifacts o history).'''
    partes = []
    # 1) mensaje de estado
    msg = (result.get("status") or {}).get("message")
    if msg:
        for p in msg.get("parts", []):
            if p.get("text"):
                partes.append(p["text"])
    # 2) artifacts
    for art in result.get("artifacts", []) or []:
        for p in art.get("parts", []):
            if p.get("text"):
                partes.append(p["text"])
    # 3) history (último turno del agente) si aún no hay texto
    if not partes:
        for h in reversed(result.get("history", []) or []):
            if h.get("role") in ("agent", "assistant"):
                for p in h.get("parts", []):
                    if p.get("text"):
                        partes.append(p["text"])
                if partes:
                    break
    # dedup conservando orden
    vistos, limpio = set(), []
    for t in partes:
        if t not in vistos:
            vistos.add(t)
            limpio.append(t)
    return "\n".join(limpio)


async def preguntar(agente, texto, contexto=None):
    '''Envía `texto` al agente (dict {base, aid}) por JSON-RPC message/send.
    Devuelve (respuesta_texto, contextId) para encadenar turnos con memoria.'''
    base, aid = agente["base"], agente["aid"]
    mensaje = {
        "role": "user",
        "messageId": uuid.uuid4().hex,
        "parts": [{"kind": "text", "text": texto}],
    }
    if contexto:
        mensaje["contextId"] = contexto
    payload = {"jsonrpc": "2.0", "id": uuid.uuid4().hex,
               "method": "message/send", "params": {"message": mensaje}}
    r = await http_a2a.post(f"{base}/a2a/{aid}", json=payload)
    data = r.json()
    if "error" in data and data["error"]:
        raise RuntimeError(f"Error A2A: {data['error']}")
    result = data.get("result", {})
    ctx = result.get("contextId") or contexto
    respuesta = _extraer_texto(result)
    if not respuesta:
        raise RuntimeError("El agente no devolvió texto. Revisa su terminal.")
    return respuesta, ctx

print("Función preguntar(...) lista.")

Función preguntar(...) lista.


## 5. (Opcional) Probar un solo agente

Cambia el nombre por uno de los que aparecieron en "Agentes disponibles".

In [7]:
NOMBRE_PRUEBA = "agente_bryhan"  # cámbialo por un nombre del catálogo

if NOMBRE_PRUEBA in agentes_por_nombre:
    respuesta, _ = await preguntar(agentes_por_nombre[NOMBRE_PRUEBA], "Hola, ¿quién eres y en qué me puedes ayudar?")
    print(respuesta)
else:
    print(f"'{NOMBRE_PRUEBA}' no está en el catálogo. Nombres: {list(agentes_por_nombre)}")

Soy el Agente profesional de Bryhan, una IA diseñada para representar su perfil y razonar desde su experiencia documentada en AI Engineering, Data Science, automatización e IA aplicada.

Puedo ayudarte a:
1. **Analizar procesos y proponer automatización o IA**: Identificar objetivos, reglas, datos y restricciones para determinar si se requiere automatización tradicional, datos, Machine Learning, IA generativa o procesamiento documental.
2. **Diseñar conceptualmente soluciones de Data e IA**: Estructurar componentes, flujos de información, integraciones, seguridad y mantenibilidad.
3. **Explicar, comparar y recomendar alternativas técnicas**: Evaluar opciones tecnológicas y metodológicas mediante criterios objetivos, entregando recomendaciones condicionadas a tu contexto.

Toda la información entregada se basa estrictamente en la trayectoria documentada de Bryhan como Ingeniero Civil Industrial especializado en AI Engineering y Data Science aplicado en la banca chilena (Banco Santander 

## 5.1 Las tres preguntas (Etapa 2)

Se envían **por A2A** (a través del túnel) tres preguntas a tu propio agente para comprobar que, también de forma remota, sus respuestas reflejan:

1. **Personalidad e identidad** — quién es y cómo se presenta.
2. **Conocimientos** — un antecedente documentado de tu perfil.
3. **Una de tus habilidades** — aplicada a un caso concreto.

Las tres comparten el mismo `contextId`, de modo que el agente conserva memoria entre ellas. Cambia `MI_NOMBRE_A2A` si el nombre de tu Agent Card es distinto.

In [8]:
MI_NOMBRE_A2A = "agente_bryhan"  # nombre de TU Agent Card en el catálogo

TRES_PREGUNTAS = [
    ("Personalidad e identidad",
     "Preséntate: ¿quién eres y en qué puedes ayudarme? No inventes antecedentes."),
    ("Conocimientos",
     "Cuéntame un caso o conocimiento documentado de tu perfil profesional y por qué es relevante."),
    ("Habilidad aplicada",
     "Aplica una de tus tres habilidades a este caso: quiero automatizar un proceso documental repetitivo. ¿Cómo lo abordarías?"),
]

if MI_NOMBRE_A2A not in agentes_por_nombre:
    print(f"'{MI_NOMBRE_A2A}' no está en el catálogo. Nombres: {list(agentes_por_nombre)}")
else:
    mi_agente = agentes_por_nombre[MI_NOMBRE_A2A]
    ctx_preguntas = None  # mismo contextId en las tres => memoria
    for etiqueta, pregunta in TRES_PREGUNTAS:
        print(f"\n{'=' * 20} {etiqueta} {'=' * 20}")
        print("Pregunta:", pregunta)
        respuesta, ctx_preguntas = await preguntar(mi_agente, pregunta, contexto=ctx_preguntas)
        print("Respuesta:", respuesta)


==================== Personalidad e identidad ====================
Pregunta: Preséntate: ¿quién eres y en qué puedes ayudarme? No inventes antecedentes.
Respuesta: Soy el Agente profesional de Bryhan, una IA diseñada para representar su perfil y razonar desde su experiencia documentada en AI Engineering, Data Science, automatización e IA aplicada.

Basado en la trayectoria documentada de Bryhan, puedo ayudarte en las siguientes áreas:

1. **Análisis de procesos y propuestas de automatización o IA**: Identificar objetivos, reglas, datos y restricciones en procesos empresariales o financieros para determinar si requieren automatización tradicional, analítica, Machine Learning, IA generativa o procesamiento documental.
2. **Diseño conceptual de soluciones de Datos e IA**: Estructurar componentes, flujos de información, integraciones y consideraciones de seguridad, basándome en su experiencia en banca chilena (Banco Santander Chile e Itaú Chile) con tecnologías como Python, Azure Databric

## 6. Conversación A → B → A

A propone tres ideas, B las mejora desde su especialidad, A integra ambos aportes en un plan. Usa los nombres exactos del catálogo.

In [9]:
NOMBRE_A = "agente_bryhan"            # tu agente
NOMBRE_B = "agente_czz"               # agente del compañero (Carlos)

A = agentes_por_nombre[NOMBRE_A]
B = agentes_por_nombre[NOMBRE_B]

aporte_a, ctx_a = await preguntar(
    A, f"Reto: {RETO}\nPropón tres ideas concretas desde tu especialidad."
)
aporte_b, ctx_b = await preguntar(
    B, f"Reto: {RETO}\nTu colega propone:\n{aporte_a}\n"
       "Mejora la propuesta desde tu especialidad y explica qué cambiarías."
)
acuerdo, ctx_a = await preguntar(
    A, f"Tu colega responde:\n{aporte_b}\n"
       "Integra ambos aportes en un plan conjunto breve.",
    contexto=ctx_a,
)

for nombre, texto in [(NOMBRE_A, aporte_a), (NOMBRE_B, aporte_b), (NOMBRE_A, acuerdo)]:
    print(f"\n===== {nombre} =====\n{texto}")


===== agente_bryhan =====
**Conclusión:** Desde la especialidad de Bryhan en IA aplicada, automatización y ciencia de datos, se proponen tres actividades concretas para la bienvenida de estudiantes nuevos que combinan la tecnología, el análisis de datos y la resolución práctica de problemas.

---

### 1. Taller práctico: "Predice tu éxito académico" (Introducción interactiva a Machine Learning)
* **Objetivo:** Demostrar de forma práctica cómo los datos y los modelos predictivos pueden anticipar escenarios reales, usando el perfil del estudiante.
* **Dinámica:** Se presenta un dataset anónimo y simplificado (basado en hábitos de estudio, tiempo de trayecto, uso de bibliotecas, etc.). Mediante una interfaz guiada en Python (o herramientas visuales de análisis), los estudiantes ajustan parámetros sencillos para predecir tasas de aprobación o uso óptimo de recursos universitarios.
* **Aporte desde el perfil:** Conecta la experiencia docente de Bryhan en datos y Machine Learning con un cas

## 7. Cerrar conexiones

Ejecuta al terminar para liberar el cliente HTTP.

In [ ]:
if "http_a2a" in globals():
    await http_a2a.aclose()
print("Conexiones cerradas.")